# M3 — traversability (PU-learning) (Marswalk)

Learns a per-cell **traversability** score from a rover's drive using Positive-Unlabeled learning: the cells the rover crossed are positive, everywhere else is unlabeled. Here the traverse is **synthetic** (a subset of truly traversable cells) so it runs offline; wire real Perseverance / Curiosity waypoints and the same code learns from real drives.

**How to run:** **Run all**, then send back **`traversability_report.json`** and **`traversability_samples.png`**.

In [ ]:
!pip install -q "git+https://github.com/MONSTER4REX/Martian-Map.git"
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', dev)

## Train on positives+unlabeled, score held-out terrain (ROC-AUC vs truth)

In [ ]:
import json, time
import numpy as np
from marswalk.ml.traversability import make_pu_dataset, train_pu, evaluate

Xtr, str_, ytr = make_pu_dataset(range(16))
Xte, ste, yte = make_pu_dataset(range(200, 206))
t0 = time.time()
model = train_pu(Xtr, str_, epochs=80, device=dev)
rep = evaluate(model, Xte, ste, yte)
secs = time.time() - t0
print(rep)

In [ ]:
report = {**rep.as_dict(), 'device': (torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU'),
          'epochs': 80, 'seconds': round(secs, 1)}
with open('traversability_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
Xo, so, yo = make_pu_dataset(range(300, 301))   # one unseen terrain
score = model.score(Xo).reshape(40, 40)
fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
ax[0].imshow(yo.reshape(40, 40), cmap='Greens'); ax[0].set_title('true traversable')
im = ax[1].imshow(score, cmap='Greens', vmin=0, vmax=1); ax[1].set_title('PU score')
for a in ax: a.set_xticks([]); a.set_yticks([])
fig.colorbar(im, ax=ax, fraction=0.025)
fig.suptitle(f"traversability from a synthetic traverse — ROC-AUC {rep.roc_auc:.3f}")
fig.savefig('traversability_samples.png', dpi=120, bbox_inches='tight')
print('wrote traversability_samples.png')

## Send back

Download **`traversability_report.json`** and **`traversability_samples.png`** and send them to the lead. Thanks!